In [1]:
# This allows us to import from all folders one level up from notebooks folder - run 1 time
import sys
from pathlib import Path

print('All paths pre-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")
print('-'*100)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print('All paths post-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")


All paths pre-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
----------------------------------------------------------------------------------------------------
All paths post-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
5: /Users/irabandutta/Developer/2026-08-llm-from-scratch


# Imports

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple
from src.model.llm_config import LLMConfig
from src.model.llm import LLM

# DataLoader

In [11]:
class TokenDataLoader:
    def __init__(self, B:int, T:int, binary_file_path:str, dtype:np.dtype, debug:bool=False):
        self.B = max(1, B)
        self.T = max(4, T)
        if not debug:
            self.tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
            print(f"Loaded {len(self.tokens)/10e6}M tokens, 1 epoch = {len(self.tokens)//(self.T)} batches")
        else:
            self.tokens = np.arange(1, 101)
        if len(self.tokens)<=(self.B)*(self.T):
            raise ValueError(
                f"Current values of batch size {B} and seq length {T} are too large for dataset, please reduce either or both"
            )
        self.curr_idx = 0

    def next_batch(self) -> Tuple[torch.tensor]:
        B, T = (self.B), (self.T)

        buffer = self.tokens[self.curr_idx:self.curr_idx+(B*T+1)]
        x = torch.tensor(buffer[:-1]).view(B, T)
        y = torch.tensor(buffer[1:]).view(B, T)

        # Covert x,y from uint16 to int32 and int64
        x = x.int()
        y = y.long()

        self.curr_idx += B*T
        if self.curr_idx+(B*T+1) > len(self.tokens):
            self.curr_idx=0

        return x, y


binary_file_path = '../data/tinystories/processed/train.bin'
B = 4
T = 32
tok_dl = TokenDataLoader(B, T, binary_file_path, np.uint16, False)
# print('-'*100)
# steps = 2
# for _ in range(steps):
#     x, y = tok_dl.next_batch()
#     print('x:\n', x)
#     print('y:\n', y)    
# print('-'*100)

Loaded 47.1872517M tokens, 1 epoch = 14746016 batches


In [4]:
# Get a sample batch of tokens of shape (B, T) and overfit model on that batch

x, y = tok_dl.next_batch()
print(x.shape, y.shape)
print(x.dtype, y.dtype)

torch.Size([4, 32]) torch.Size([4, 32])
torch.int32 torch.int64


# Instantiate model

In [5]:
# ======== DEFINE Model ========
ctx_len = T
d_model = 256

llm_config = LLMConfig(
    vocab_size=50257,
    ctx_len=ctx_len,
    d_model=d_model, 
    n_layer=4,
    ff_ratio=4,
    dropout=0.0,
    eps=1e-5,
    position_embedding='sinusoidal',
    rotary_embedding=False,
    attention='mha',
    normalization='layernorm',
    n_heads=4, 
    n_groups=None,
    use_flash=False, 
    attn_debug=False
)
print(llm_config)
print('-'*50)

# Detect and resolve device
device = 'cpu'
if torch.cuda.is_available():
    device='cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device='mps'

print("Device found:", device)
print('-'*50)


# Instantiate model
model = LLM(llm_config)
model.eval()
model = model.to(device)
print(f"Model instantiated and moved to {device}")
print('-'*50)

LLMConfig(vocab_size=50257, ctx_len=32, d_model=256, n_layer=4, ff_ratio=4, dropout=0.0, eps=1e-05, position_embedding='sinusoidal', rotary_embedding=False, attention='mha', normalization='layernorm', n_heads=4, n_groups=None, use_flash=False, attn_debug=False)
--------------------------------------------------
Device found: mps
--------------------------------------------------
Model instantiated and moved to mps
--------------------------------------------------


In [6]:
# Move x and y to device
print(x.device, y.device)
x, y = x.to(device), y.to(device)
print(x.device, y.device)
print(x.dtype, y.dtype)

cpu cpu
mps:0 mps:0
torch.int32 torch.int64


In [7]:
# 1 forward pass - track init loss
logits, loss = model(x, y)
print(logits.shape)
print(loss)

torch.Size([4, 32, 50257])
tensor(11.0608, device='mps:0', grad_fn=<NllLossBackward0>)


In [8]:
# Expected Loss at inti
-torch.log(torch.tensor(1/50257))

tensor(10.8249)

In [9]:
# Run a train loop over sample batch

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

steps = 50
for i in range(steps):
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()
    print(f"Step: {i+1}: Loss: {loss.item()}")



Step: 1: Loss: 11.060846328735352
Step: 2: Loss: 10.423377990722656
Step: 3: Loss: 10.154590606689453
Step: 4: Loss: 9.880345344543457
Step: 5: Loss: 9.706233978271484
Step: 6: Loss: 9.530862808227539
Step: 7: Loss: 9.415725708007812
Step: 8: Loss: 9.288908004760742
Step: 9: Loss: 9.198537826538086
Step: 10: Loss: 9.089189529418945
Step: 11: Loss: 9.003090858459473
Step: 12: Loss: 8.893926620483398
Step: 13: Loss: 8.800617218017578
Step: 14: Loss: 8.681468963623047
Step: 15: Loss: 8.573681831359863
Step: 16: Loss: 8.437540054321289
Step: 17: Loss: 8.310349464416504
Step: 18: Loss: 8.153005599975586
Step: 19: Loss: 8.006157875061035
Step: 20: Loss: 7.830563545227051
Step: 21: Loss: 7.67388916015625
Step: 22: Loss: 7.494692325592041
Step: 23: Loss: 7.345759868621826
Step: 24: Loss: 7.178132057189941
Step: 25: Loss: 7.042081832885742
Step: 26: Loss: 6.882050037384033
Step: 27: Loss: 6.747413635253906
Step: 28: Loss: 6.584256172180176
Step: 29: Loss: 6.4443817138671875
Step: 30: Loss: 6.27